# Walkthrough: from a knowledge graph to an approved plan

A guided tour through the demo, one concept per cell. It runs without an API key: the agents use the scripted `OfflineModel`, everything else (SPARQL, SHACL, LangGraph) is real.

The task: *Level transmitter type A is discontinued. Find all installation points in the plant and prepare the replacement with type B.*

## 1. The knowledge graph and a SPARQL query

`load_graph()` reads the ontology (`ontology/plant.ttl`) and the plant data (`data/plant.ttl`) into one rdflib graph. The query `installation-points` finds every device of a type, with its location and Ex zone ([chapter 01](../docs/concepts/01-ontology-and-sparql.md)).

In [1]:
from engineering_agents.cli import format_result, format_table
from engineering_agents.graph import load_graph
from engineering_agents.queries import run_query

graph = load_graph()
print(f"{len(graph)} triples\n")
print(format_table(run_query(graph, "installation-points", "LevelTransmitterA")))

299 triples

device     location                    unit            zone           medium
---------  --------------------------  --------------  -------------  --------------------
ex:LT-101  T-101 Solvent storage tank  U100 Tank farm  Zone 1         solvent
ex:LT-102  T-102 Process water tank    U100 Tank farm  Non-hazardous  process water
ex:LT-103  T-103 Product tank          U100 Tank farm  Zone 2         intermediate product
ex:LT-301  B-301 Cooling water basin   U300 Utilities  Non-hazardous  cooling water


## 2. The SHACL check on plan B

Replacing every type A device with type B is checked against the shapes in `shapes/plant-shapes.ttl` ([chapter 02](../docs/concepts/02-shacl.md)). Type B is not approved for Zone 1, so LT-101 breaks the rule.

In [2]:
from engineering_agents.validation import check_replacements

plan_b = {f"LT-{n}": "LevelTransmitterB" for n in (101, 102, 103, 301)}
print(format_result(check_replacements(graph, plan_b)))

Conforms: no (1 violation)

device     location  zone         type
---------  --------  -----------  --------------------
ex:LT-101  ex:T-101  plant:Zone1  ex:LevelTransmitterB

- LT-101 at T-101 Solvent storage tank (Zone 1): type Level transmitter type B is not approved for Zone 1.


## 3. The LangGraph graph

The supervisor routes between the two agents, the validator and the approval; every node reports back to it ([chapter 04](../docs/concepts/04-orchestration-with-langgraph.md)). The graph draws itself as a Mermaid diagram (rendered by JupyterLab and GitHub).

In [3]:
from IPython.display import Markdown

from engineering_agents.orchestration import build_graph

Markdown(f"```mermaid\n{build_graph().get_graph().draw_mermaid()}\n```")

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	analysis(analysis)
	planning(planning)
	validator(validator)
	approval(approval)
	hand_over(hand_over)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analysis --> supervisor;
	approval --> supervisor;
	planning --> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> analysis;
	supervisor -.-> approval;
	supervisor -.-> hand_over;
	supervisor -.-> planning;
	supervisor -.-> validator;
	validator --> supervisor;
	hand_over --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## 4. Run the graph up to the approval

A run that can pause needs a **checkpointer** (it saves the state after every step) and a **thread id** (the key the checkpoints are saved under). The run stops at the `approval` node's `interrupt(...)`; `invoke` returns the state so far, with the interrupt's request under `__interrupt__` ([chapter 05](../docs/concepts/05-human-in-the-loop.md)).

In [4]:
from engineering_agents.offline import OfflineModel
from engineering_agents.orchestration import Context, initial_state, memory_checkpointer

app = build_graph(checkpointer=memory_checkpointer())
config = {"configurable": {"thread_id": "walkthrough"}}
context = Context(model=OfflineModel())

state = app.invoke(initial_state(), config, context=context)
for entry in state["trace"]:
    print(f"[{entry['node']}] {entry['note']}")
print()
print(state["__interrupt__"][0].value)

[supervisor] -> analysis: no analysis yet; find the affected devices first
[analysis] found 4 device(s): ex:LT-101 (Zone 1), ex:LT-102 (Non-hazardous), ex:LT-103 (Zone 2), ex:LT-301 (Non-hazardous)
[supervisor] -> planning: the affected devices are known; a plan is needed
[planning] proposed a plan (round 1)
[supervisor] -> validator: a new plan must be checked against the rules
[validator] does not conform: 1 violation(s)
[supervisor] -> planning: the plan breaks 1 rule(s); send it back with them
[planning] revised the plan (round 2)
[supervisor] -> validator: a new plan must be checked against the rules
[validator] conforms
[supervisor] -> approval: the plan conforms to all rules; a person must approve it

Approval needed: replacement plan, round 2, 4 device(s)
  ex:LT-101  Zone 1         ex:LevelTransmitterA -> ex:LevelTransmitterC
  ex:LT-102  Non-hazardous  ex:LevelTransmitterA -> ex:LevelTransmitterB
  ex:LT-103  Zone 2         ex:LevelTransmitterA -> ex:LevelTransmitterB
  ex:LT

## 5. Inspect the paused run

The run is not waiting in memory as a running function: it has ended, and its state sits in the checkpointer under the thread id. `get_state` reads it back. `next` names the node that will run on resume.

In [5]:
paused = app.get_state(config)
print("next node:      ", paused.next)
print("status:         ", paused.values["status"])
print("planning rounds:", paused.values["planning_rounds"])
print("proposed types: ", paused.values["plan"].replacements())

next node:       ('approval',)
status:          conforms
planning rounds: 2
proposed types:  {'ex:LT-101': 'ex:LevelTransmitterC', 'ex:LT-102': 'ex:LevelTransmitterB', 'ex:LT-103': 'ex:LevelTransmitterB', 'ex:LT-301': 'ex:LevelTransmitterB'}


## 6. Resume with an approval

`Command(resume=...)` on the same thread continues the run. The approval node runs again from its first line; this time `interrupt` returns the decision, and the supervisor routes to `hand_over`.

In [6]:
from langgraph.types import Command

state = app.invoke(Command(resume={"approved": True}), config, context=context)
for entry in state["trace"][-3:]:
    print(f"[{entry['node']}] {entry['note']}")
print("\nstatus:", state["status"])

[approval] approved by the reviewer
[supervisor] -> hand_over: the reviewer approved the plan
[hand_over] plan with 4 entries handed over

status: handed_over


## Try it yourself

- Rerun cells 4 and 5 with a new thread id, then resume with `{"approved": False, "comment": "..."}`. Where does the run stop next, and what does `state["review_comment"]` hold?
- Replace `OfflineModel()` with `make_model()` from `engineering_agents.agent` to let the OpenAI model play the agents (needs `OPENAI_API_KEY`).
- In a terminal: `uv run engineering-agents demo` runs the same scenario with narrated output and asks for the approval there.